In [2]:

import pandas as pd
from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

ATTENDANCE_FILE = Path(
    "output/second_reports/attendance_feature_engineered.xlsx"
)

CALENDAR_FILE = Path(
    "output/dim_calendar.csv"
)

OUTPUT_DIR = Path("output/master")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "master_attendance_weekly.csv"
QUALITY_FILE = OUTPUT_DIR / "master_attendance_weekly_quality_report.csv"

WEEK_KEYS = [
    "week_year",
    "week_start",
    "week_end"
]

REQUIRED_ATTENDANCE_COLUMNS = [
    "employee_id",
    "attendance_date",
    "cost_center_code",
    "cost_center_type"
]

REQUIRED_CALENDAR_COLUMNS = [
    "attendance_date",
    "week_start",
    "week_end",
    "week_year",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves",
    "ho_weekly_attendance_quota",
    "non_ho_weekly_attendance_quota"
]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def convert_boolean(series):
    """
    Convert common TRUE/FALSE representations into
    pandas nullable boolean.
    """

    true_values = {"TRUE", "1", "YES", "Y"}
    false_values = {"FALSE", "0", "NO", "N"}

    def convert_value(value):

        if pd.isna(value):
            return pd.NA

        value = str(value).strip().upper()

        if value in true_values:
            return True

        if value in false_values:
            return False

        return pd.NA

    return series.apply(convert_value).astype("boolean")


def is_nonempty(series):
    """
    Identify values that are not empty or missing.
    """

    text = series.astype("string").str.strip()

    return (
        series.notna()
        & text.ne("")
        & ~text.str.lower().isin(
            ["nan", "none", "nat", "<na>"]
        )
    ).fillna(False)


def unique_join(series):
    """
    Preserve unique non-empty values as a single
    pipe-separated string.
    """

    values = []

    for value in series:

        if pd.isna(value):
            continue

        value = str(value).strip()

        if value == "" or value.lower() in [
            "nan", "none", "nat", "<na>"
        ]:
            continue

        if value not in values:
            values.append(value)

    return " | ".join(values)


def first_non_null(series):
    """
    Return the first non-null value.
    """

    values = series.dropna()

    if len(values) == 0:
        return pd.NA

    return values.iloc[0]


def validate_weekly_constant(data, column):
    """
    Ensure one weekly calendar value is consistent
    across all dates belonging to the same week.
    """

    counts = (
        data.groupby(WEEK_KEYS, dropna=False)[column]
        .nunique(dropna=True)
    )

    if (counts > 1).any():

        raise ValueError(
            f"Inconsistent {column} values found "
            "within the same calendar week."
        )


# ============================================================
# 3. LOAD DATA
# ============================================================

print("Loading attendance data...")

attendance = pd.read_excel(
    ATTENDANCE_FILE,
    sheet_name=0
)

print("Loading calendar data...")

calendar = pd.read_csv(
    CALENDAR_FILE,
    low_memory=False
)

attendance.columns = attendance.columns.str.strip()
calendar.columns = calendar.columns.str.strip()

print("Attendance rows:", len(attendance))
print("Calendar rows:", len(calendar))


# ============================================================
# 4. VALIDATE REQUIRED COLUMNS
# ============================================================

missing_attendance_columns = [
    col for col in REQUIRED_ATTENDANCE_COLUMNS
    if col not in attendance.columns
]

missing_calendar_columns = [
    col for col in REQUIRED_CALENDAR_COLUMNS
    if col not in calendar.columns
]

if missing_attendance_columns:

    raise ValueError(
        f"Missing attendance columns: "
        f"{missing_attendance_columns}"
    )

if missing_calendar_columns:

    raise ValueError(
        f"Missing calendar columns: "
        f"{missing_calendar_columns}"
    )


# ============================================================
# 5. NORMALIZE DATE COLUMNS
# ============================================================

attendance["attendance_date"] = pd.to_datetime(
    attendance["attendance_date"],
    errors="coerce"
).dt.normalize()

calendar["attendance_date"] = pd.to_datetime(
    calendar["attendance_date"],
    errors="coerce"
).dt.normalize()

calendar["week_start"] = pd.to_datetime(
    calendar["week_start"],
    errors="coerce"
).dt.normalize()

calendar["week_end"] = pd.to_datetime(
    calendar["week_end"],
    errors="coerce"
).dt.normalize()

if attendance["attendance_date"].isna().any():

    raise ValueError(
        "Attendance contains invalid attendance_date values."
    )

if calendar["attendance_date"].isna().any():

    raise ValueError(
        "Calendar contains invalid attendance_date values."
    )

if calendar[WEEK_KEYS].isna().any().any():

    raise ValueError(
        "Calendar contains missing weekly identifiers."
    )


# ============================================================
# 6. NORMALIZE CALENDAR BOOLEAN COLUMNS
# ============================================================

BOOLEAN_CALENDAR_COLUMNS = [
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave"
]

for col in BOOLEAN_CALENDAR_COLUMNS:

    calendar[col] = convert_boolean(calendar[col])

    if calendar[col].isna().any():

        raise ValueError(
            f"Calendar contains missing or invalid "
            f"values in {col}."
        )


# ============================================================
# 7. VALIDATE CALENDAR
# ============================================================

if calendar["attendance_date"].duplicated().any():

    duplicate_dates = calendar.loc[
        calendar["attendance_date"].duplicated(keep=False),
        "attendance_date"
    ].drop_duplicates()

    raise ValueError(
        "Calendar contains duplicate attendance_date values:\n"
        f"{duplicate_dates.head(10).to_list()}"
    )


# Ensure weekly quotas are consistent across calendar dates.

for col in [
    "ho_weekly_attendance_quota",
    "non_ho_weekly_attendance_quota",
    "weekday_national_holidays",
    "weekday_joint_leaves"
]:

    calendar[col] = pd.to_numeric(
        calendar[col],
        errors="coerce"
    )

    validate_weekly_constant(calendar, col)


# ============================================================
# 8. BUILD WEEKLY CALENDAR SUMMARY
# ============================================================

print("Building weekly calendar summary...")

calendar["national_holiday_date"] = (
    calendar["attendance_date"].where(
        calendar["is_national_holiday"]
    )
)

calendar["joint_leave_date"] = (
    calendar["attendance_date"].where(
        calendar["is_joint_leave"]
    )
)

calendar_weekly = (
    calendar.groupby(
        WEEK_KEYS,
        dropna=False
    )
    .agg(
        ho_weekly_attendance_quota=(
            "ho_weekly_attendance_quota",
            "first"
        ),

        non_ho_weekly_attendance_quota=(
            "non_ho_weekly_attendance_quota",
            "first"
        ),

        weekday_national_holidays=(
            "weekday_national_holidays",
            "first"
        ),

        weekday_joint_leaves=(
            "weekday_joint_leaves",
            "first"
        ),

        calendar_days_in_week=(
            "attendance_date",
            "nunique"
        ),

        calendar_weekend_days=(
            "is_weekend",
            "sum"
        ),

        calendar_national_holiday_days=(
            "is_national_holiday",
            "sum"
        ),

        calendar_joint_leave_days=(
            "is_joint_leave",
            "sum"
        ),

        national_holiday_dates=(
            "national_holiday_date",
            unique_join
        ),

        joint_leave_dates=(
            "joint_leave_date",
            unique_join
        )
    )
    .reset_index()
)


# ============================================================
# 9. CLASSIFY HO AND NON-HO
# ============================================================

print("Classifying employee categories...")

# Cost center code is the primary classification.
# cost_center_type is used only when the code is missing.

cost_center_code = (
    attendance["cost_center_code"]
    .astype("string")
    .str.strip()
)

code_available = (
    is_nonempty(cost_center_code)
)

code_is_ho = cost_center_code.str.startswith(
    "9",
    na=False
)

type_is_ho = (
    attendance["cost_center_type"]
    .astype("string")
    .str.strip()
    .eq("FIXED_5")
    .fillna(False)
)

type_is_non_ho = (
    attendance["cost_center_type"]
    .astype("string")
    .str.strip()
    .eq("NON_FIXED_6")
    .fillna(False)
)

attendance["is_ho_employee"] = pd.Series(
    pd.NA,
    index=attendance.index,
    dtype="boolean"
)

attendance.loc[
    code_available,
    "is_ho_employee"
] = code_is_ho.loc[code_available]

attendance.loc[
    ~code_available & type_is_ho,
    "is_ho_employee"
] = True

attendance.loc[
    ~code_available & type_is_non_ho,
    "is_ho_employee"
] = False

if attendance["is_ho_employee"].isna().any():

    raise ValueError(
        "Some attendance rows cannot be classified "
        "as HO or non-HO. Check cost_center_code "
        "and cost_center_type."
    )

recognized_type = attendance[
    "cost_center_type"
].isin(["FIXED_5", "NON_FIXED_6"])

attendance["cost_center_classification_conflict"] = (
    recognized_type
    & (
        code_is_ho
        != type_is_ho
    )
)


# ============================================================
# 10. PREPARE ATTENDANCE ACTIVITY
# ============================================================

print("Preparing attendance activity...")

# Use existing engineered flags when available.
# Also check the actual clock columns to avoid
# overlooking recorded attendance activity.

if "has_clock_in" in attendance.columns:

    has_clock_in = convert_boolean(
        attendance["has_clock_in"]
    ).fillna(False)

else:

    if "clock_in_time" in attendance.columns:

        has_clock_in = is_nonempty(
            attendance["clock_in_time"]
        )

    else:

        has_clock_in = pd.Series(
            False,
            index=attendance.index
        )


if "has_clock_out" in attendance.columns:

    has_clock_out = convert_boolean(
        attendance["has_clock_out"]
    ).fillna(False)

else:

    if "clock_out_time" in attendance.columns:

        has_clock_out = is_nonempty(
            attendance["clock_out_time"]
        )

    else:

        has_clock_out = pd.Series(
            False,
            index=attendance.index
        )


if "has_attendance_activity" in attendance.columns:

    existing_activity = convert_boolean(
        attendance["has_attendance_activity"]
    ).fillna(False)

else:

    existing_activity = pd.Series(
        False,
        index=attendance.index
    )


attendance["_has_clock_in"] = has_clock_in
attendance["_has_clock_out"] = has_clock_out

attendance["_has_activity"] = (
    existing_activity
    | has_clock_in
    | has_clock_out
)


# ============================================================
# 11. JOIN ATTENDANCE WITH CALENDAR
# ============================================================

print("Joining attendance with calendar...")

calendar_join = calendar[
    [
        "attendance_date",
        "week_year",
        "week_start",
        "week_end"
    ]
].copy()

attendance = attendance.merge(
    calendar_join,
    on="attendance_date",
    how="left",
    validate="many_to_one",
    indicator=True
)

unmatched_calendar_rows = (
    attendance["_merge"] != "both"
).sum()

if unmatched_calendar_rows > 0:

    raise ValueError(
        f"{unmatched_calendar_rows} attendance rows "
        "do not have a matching calendar date. "
        "Check the calendar date coverage."
    )

attendance.drop(
    columns=["_merge"],
    inplace=True
)


# ============================================================
# 12. CREATE EMPLOYEE GROUPING KEY
# ============================================================

# Normally employee_id is the grouping identifier.
# If employee_id is missing, use employee_name.
# If both are missing, keep the row separate.

attendance["_employee_group_key"] = (
    attendance["employee_id"]
    .astype("string")
    .str.strip()
)

missing_id = (
    ~is_nonempty(attendance["employee_id"])
)

if "employee_name" in attendance.columns:

    name_available = is_nonempty(
        attendance["employee_name"]
    )

    attendance.loc[
        missing_id & name_available,
        "_employee_group_key"
    ] = (
        "NAME:"
        + attendance.loc[
            missing_id & name_available,
            "employee_name"
        ].astype("string").str.strip()
    )

missing_both = (
    ~is_nonempty(attendance["employee_id"])
    & (
        ~is_nonempty(attendance["employee_name"])
        if "employee_name" in attendance.columns
        else True
    )
)

attendance.loc[
    missing_both,
    "_employee_group_key"
] = (
    "UNKNOWN_ROW:"
    + attendance.index[missing_both].astype(str)
)


# ============================================================
# 13. PREPARE WEEKLY ACTIVITY DATES
# ============================================================

attendance["_activity_date"] = (
    attendance["attendance_date"].where(
        attendance["_has_activity"]
    )
)

attendance["_clock_in_date"] = (
    attendance["attendance_date"].where(
        attendance["_has_clock_in"]
    )
)

attendance["_clock_out_date"] = (
    attendance["attendance_date"].where(
        attendance["_has_clock_out"]
    )
)

attendance["_attendance_date_text"] = (
    attendance["attendance_date"].dt.strftime(
        "%Y-%m-%d"
    )
)



# ============================================================
# 14. AGGREGATE ATTENDANCE INTO EMPLOYEE-WEEK
# ============================================================

print("Aggregating attendance into weekly records...")

GROUP_KEYS = [
    "_employee_group_key",
    "week_year",
    "week_start",
    "week_end"
]

# Columns that are already handled by dedicated aggregations.
EXCLUDED_AGGREGATION_COLUMNS = [
    "is_ho_employee",
    "cost_center_classification_conflict"
]

# Preserve all original attendance columns,
# except columns handled separately below.

original_attendance_columns = [
    col
    for col in attendance.columns
    if not col.startswith("_")
    and col not in WEEK_KEYS
    and col != "attendance_date"
    and col not in EXCLUDED_AGGREGATION_COLUMNS
]

# Aggregate original attendance attributes.
attendance_aggregation = {
    col: unique_join
    for col in original_attendance_columns
}

# Build weekly attendance aggregation.
weekly_attendance = (
    attendance.groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        **{
            col: (col, func)
            for col, func in attendance_aggregation.items()
        },

        attendance_dates=(
            "_attendance_date_text",
            unique_join
        ),

        attendance_row_count=(
            "attendance_date",
            "size"
        ),

        attendance_activity_days=(
            "_activity_date",
            "nunique"
        ),

        clock_in_days=(
            "_clock_in_date",
            "nunique"
        ),

        clock_out_days=(
            "_clock_out_date",
            "nunique"
        ),

        ho_classification_count=(
            "is_ho_employee",
            "nunique"
        ),

        is_ho_employee=(
            "is_ho_employee",
            "first"
        ),

        classification_conflict_count=(
            "cost_center_classification_conflict",
            "sum"
        )
    )
    .reset_index()
)

print("Weekly attendance rows:", len(weekly_attendance))


# ============================================================
# 15. JOIN WEEKLY CALENDAR INFORMATION
# ============================================================

print("Joining weekly calendar information...")

master_weekly = weekly_attendance.merge(
    calendar_weekly,
    on=WEEK_KEYS,
    how="left",
    validate="many_to_one",
    indicator=True
)

unmatched_week_rows = (
    master_weekly["_merge"] != "both"
).sum()

if unmatched_week_rows > 0:

    raise ValueError(
        f"{unmatched_week_rows} employee-week records "
        "do not have a matching weekly calendar."
    )

master_weekly.drop(
    columns=["_merge"],
    inplace=True
)


# ============================================================
# 16. DETERMINE WEEKLY EMPLOYEE CLASSIFICATION
# ============================================================

# If an employee has different HO classifications
# within the same week, flag the record for review.

master_weekly["weekly_classification_conflict"] = (
    master_weekly["ho_classification_count"] > 1
)

master_weekly["is_ho_employee"] = (
    master_weekly["is_ho_employee"].astype("boolean")
)

master_weekly.loc[
    master_weekly["weekly_classification_conflict"],
    "is_ho_employee"
] = pd.NA


# ============================================================
# 17. DETERMINE WEEKLY ATTENDANCE QUOTA
# ============================================================

master_weekly["weekly_attendance_quota"] = pd.Series(
    pd.NA,
    index=master_weekly.index,
    dtype="Float64"
)

ho_mask = (
    master_weekly["is_ho_employee"] == True
)

non_ho_mask = (
    master_weekly["is_ho_employee"] == False
)

master_weekly.loc[
    ho_mask,
    "weekly_attendance_quota"
] = master_weekly.loc[
    ho_mask,
    "ho_weekly_attendance_quota"
]

master_weekly.loc[
    non_ho_mask,
    "weekly_attendance_quota"
] = master_weekly.loc[
    non_ho_mask,
    "non_ho_weekly_attendance_quota"
]


# ============================================================
# 18. CALCULATE WEEKLY ATTENDANCE GAP
# ============================================================

master_weekly["weekly_attendance_gap"] = (
    master_weekly["attendance_activity_days"]
    - master_weekly["weekly_attendance_quota"]
)


# ============================================================
# 19. WEEKLY ATTENDANCE STATUS
# ============================================================

master_weekly["weekly_attendance_status"] = (
    "CLASSIFICATION_REVIEW"
)

valid_quota = (
    master_weekly["weekly_attendance_quota"].notna()
)

master_weekly.loc[
    valid_quota
    & (
        master_weekly["weekly_attendance_gap"] < 0
    ),
    "weekly_attendance_status"
] = "BELOW_WEEKLY_QUOTA"

master_weekly.loc[
    valid_quota
    & (
        master_weekly["weekly_attendance_gap"] == 0
    ),
    "weekly_attendance_status"
] = "MEETS_WEEKLY_QUOTA"

master_weekly.loc[
    valid_quota
    & (
        master_weekly["weekly_attendance_gap"] > 0
    ),
    "weekly_attendance_status"
] = "ABOVE_WEEKLY_QUOTA"


# Weekly comparison is applicable to both employee groups.
# This does NOT imply that daily absence can be determined.

master_weekly["is_weekly_quota_applicable"] = (
    valid_quota
)


# ============================================================
# 20. ADD DATA QUALITY FLAGS
# ============================================================

master_weekly["has_attendance_activity_in_week"] = (
    master_weekly["attendance_activity_days"] > 0
)

master_weekly["has_classification_conflict"] = (
    master_weekly["weekly_classification_conflict"]
    | (
        master_weekly["classification_conflict_count"] > 0
    )
)


# ============================================================
# 21. FINAL COLUMN ORGANIZATION
# ============================================================

# Original attendance attributes remain in the output.
# Weekly calendar attributes and weekly metrics are added.

final_columns = [
    col for col in master_weekly.columns
    if col != "_employee_group_key"
]

master_weekly = master_weekly[
    final_columns
]


# ============================================================
# 22. QUALITY REPORT
# ============================================================

print("Creating quality report...")

quality_report = pd.DataFrame({
    "metric": [
        "attendance_source_rows",
        "calendar_source_rows",
        "weekly_master_rows",
        "unique_employees",
        "HO_employee_week_records",
        "non_HO_employee_week_records",
        "weekly_classification_conflicts",
        "cost_center_classification_conflicts",
        "attendance_rows_with_missing_employee_id",
        "employee_weeks_with_activity",
        "employee_weeks_without_activity",
        "employee_weeks_below_quota",
        "employee_weeks_meeting_quota",
        "employee_weeks_above_quota",
        "employee_weeks_without_valid_quota",
        "total_attendance_activity_days"
    ],

    "value": [
        len(attendance),
        len(calendar),
        len(master_weekly),

        master_weekly[
            "employee_id"
        ].nunique(),

        int(ho_mask.sum()),

        int(non_ho_mask.sum()),

        int(
            master_weekly[
                "weekly_classification_conflict"
            ].sum()
        ),

        int(
            (
                master_weekly[
                    "classification_conflict_count"
                ] > 0
            ).sum()
        ),

        int(
            (~is_nonempty(attendance["employee_id"])).sum()
        ),

        int(
            master_weekly[
                "has_attendance_activity_in_week"
            ].sum()
        ),

        int(
            (
                ~master_weekly[
                    "has_attendance_activity_in_week"
                ]
            ).sum()
        ),

        int(
            (
                master_weekly[
                    "weekly_attendance_status"
                ] == "BELOW_WEEKLY_QUOTA"
            ).sum()
        ),

        int(
            (
                master_weekly[
                    "weekly_attendance_status"
                ] == "MEETS_WEEKLY_QUOTA"
            ).sum()
        ),

        int(
            (
                master_weekly[
                    "weekly_attendance_status"
                ] == "ABOVE_WEEKLY_QUOTA"
            ).sum()
        ),

        int(
            master_weekly[
                "weekly_attendance_quota"
            ].isna().sum()
        ),

        int(
            master_weekly[
                "attendance_activity_days"
            ].sum()
        )
    ]
})


# ============================================================
# 23. SAVE OUTPUT
# ============================================================

print("Saving weekly master data...")

master_weekly.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

quality_report.to_csv(
    QUALITY_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("\n========================================")
print("MASTER ATTENDANCE WEEKLY COMPLETED")
print("========================================")

print("Attendance source rows :", len(attendance))
print("Weekly master rows     :", len(master_weekly))
print("Unique employees       :", master_weekly["employee_id"].nunique())

print("\nOutput files:")
print(OUTPUT_FILE)
print(QUALITY_FILE)

print("\nWeekly attendance status:")
print(
    master_weekly[
        "weekly_attendance_status"
    ].value_counts(dropna=False)
)

print("\n========================================")


Loading attendance data...
Loading calendar data...
Attendance rows: 27711
Calendar rows: 371
Building weekly calendar summary...
Classifying employee categories...
Preparing attendance activity...
Joining attendance with calendar...


C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_4536\1092186653.py:581: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  attendance.drop(


Aggregating attendance into weekly records...
Weekly attendance rows: 10681
Joining weekly calendar information...
Creating quality report...
Saving weekly master data...


C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_4536\1092186653.py:790: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  master_weekly.drop(



MASTER ATTENDANCE WEEKLY COMPLETED
Attendance source rows : 27711
Weekly master rows     : 10681
Unique employees       : 5729

Output files:
output\master\master_attendance_weekly.csv
output\master\master_attendance_weekly_quality_report.csv

Weekly attendance status:
weekly_attendance_status
BELOW_WEEKLY_QUOTA    10677
MEETS_WEEKLY_QUOTA        4
Name: count, dtype: int64

